# DART JAX DenoiserMLP TPU checks

This notebook runs the JAX DenoiserMLP and diffusion `q_sample` checks on Kaggle TPU v5e-8.

Upload/mount these inputs:

- the DART repo snapshot containing `jax_dart/`
- a Torch-exported denoiser parity snapshot `.npz` made locally with `--mode torch-export --torch-device cpu`
- optionally, the TPU shard dataset with `metadata.json` and `shards/` for path sanity checks

The current production denoiser checkpoints in this repo are Transformer denoisers. This notebook is for the newly ported MLP denoiser path and the forward noising utility.

In [ ]:
# Edit these paths for your Kaggle mount names.
REPO_IN = "/kaggle/input/models/markuskamsties/codes/jax/default/1"
REPO = "/kaggle/working/DART"

# CPU Torch-exported snapshot from your local DART env.
SNAPSHOT = "/kaggle/input/datasets/markuskamsties/denoiser-parity/denoiser_torch_real_batch_random_mlp_cpu.npz"

# Optional shard root for path checks only. Leave empty if not mounted in this notebook.
ROOT = "/kaggle/input/datasets/markuskamsties/mp_h2_f8_r4_train_128k"

OUT_DIR = "/kaggle/working/parity_outputs"
OUT_COMPARE = f"{OUT_DIR}/denoiser_compare_real_batch_random_mlp_cpu.npz"

import os
os.environ.update({
    "REPO_IN": REPO_IN,
    "REPO": REPO,
    "SNAPSHOT": SNAPSHOT,
    "ROOT": ROOT,
    "OUT_DIR": OUT_DIR,
    "OUT_COMPARE": OUT_COMPARE,
})

print("REPO_IN:", REPO_IN)
print("SNAPSHOT:", SNAPSHOT)
print("ROOT:", ROOT)
print("OUT_COMPARE:", OUT_COMPARE)

## Copy repo to writable working space

In [ ]:
!rm -rf "$REPO"
!cp -r "$REPO_IN" "$REPO"
%cd /kaggle/working/DART
!ls -lah jax_dart/kaggle
!ls -lah jax_dart/models | grep denoiser || true
!mkdir -p "$OUT_DIR"

## Optional dependency refresh

In [ ]:
# Uncomment only if the Kaggle image does not already expose the expected TPU JAX stack.
# !pip install -q -U "jax[tpu]" flax optax orbax-checkpoint -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

## TPU/path check

In [ ]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --check-only \
  --snapshot "$SNAPSHOT" \
  --root "$ROOT"

## JAX-only parity compare

In [ ]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --task parity \
  --mode jax-compare \
  --snapshot "$SNAPSHOT" \
  --atol 1e-4 \
  --rtol 1e-4 \
  --fail-on-mismatch \
  --out-npz "$OUT_COMPARE"

## DenoiserMLP TPU smoke

In [ ]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --task smoke \
  --batch-size 8 \
  --h-dim 512 \
  --clip-dim 512 \
  --history-shape "2 276" \
  --noise-shape "1 256" \
  --n-blocks 2

## Inspect compare output

In [ ]:
import os
import numpy as np

with np.load(os.environ["OUT_COMPARE"]) as data:
    print("arrays:", sorted(data.files))
    for name in ["x_t", "denoised"]:
        torch_value = data[f"torch_{name}"]
        jax_value = data[f"jax_{name}"]
        diff = np.abs(torch_value - jax_value)
        print(name, "shape", torch_value.shape, "max_abs", diff.max(), "mean_abs", diff.mean())

## Package outputs

In [ ]:
!cd /kaggle/working && zip -r dart_jax_denoiser_mlp_tpu_outputs.zip parity_outputs